In [1]:
import ssl
import certifi

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

In [2]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [4]:
os.makedirs("../data", exist_ok=True)
os.makedirs("../checkpoints", exist_ok=True)
os.makedirs("../results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


In [5]:
import torchvision
import sklearn
import pandas

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)

torch: 2.10.0
torchvision: 0.25.0
numpy: 2.4.2
sklearn: 1.8.0
pandas: 3.0.1


In [6]:
image_size = 224

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
train_dataset = datasets.Flowers102(
    root="../data",
    split="train",
    download=False,
    transform=eval_transform
)

val_dataset = datasets.Flowers102(
    root="../data",
    split="val",
    download=False,
    transform=eval_transform
)

test_dataset = datasets.Flowers102(
    root="../data",
    split="test",
    download=False,
    transform=eval_transform
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 1020
Val size: 1020
Test size: 6149


In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Dataloaders created successfully.")

In [9]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:", labels[:10].tolist())

NameError: name 'train_loader' is not defined

In [ ]:
inv_normalize = transforms.Normalize(
    mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
    std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
)

def show_tensor_image(img_tensor, ax=None, title=None):
    img = inv_normalize(img_tensor).clamp(0, 1)
    img = img.permute(1, 2, 0).cpu().numpy()
    
    if ax is None:
        plt.imshow(img)
        if title is not None:
            plt.title(title)
        plt.axis("off")
    else:
        ax.imshow(img)
        if title is not None:
            ax.set_title(title)
        ax.axis("off")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax in axes.flatten():
    idx = random.randint(0, len(train_dataset) - 1)
    img, label = train_dataset[idx]
    show_tensor_image(img, ax=ax, title=f"Label: {label}")

plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    show_tensor_image(images[i], ax=ax, title=f"Label: {labels[i].item()}")

plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    show_tensor_image(images[i], ax=ax, title=f"Label: {labels[i].item()}")

plt.tight_layout()
plt.savefig("../figures/flowers102_samples.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved figure to ../figures/flowers102_samples.png")